# Advanced regression models for association analysis with individual level data

## Description

This notebook performs various advanced statistical analysis on multiple xQTL in a given region. Current procedures implemented include:

1. Univariate analysis
    - SuSiE
    - Univeriate TWAS weights: LASSO, Elastic net, mr.ash, SuSiE, Bayes alphabet soup
    - Cross validation of TWAS methods (optional but highly recommended if TWAS weights are computed)
2. Functional data (epigenomic xQTL) analysis
    - fSuSiE
3. Multivariate analysis
    - mvSuSiE
    - Multivariate TWAS weights: mvSuSiE and mr.mash

## Input

1. A list of regions to be analyzed (optional); the last column of this file should be region name.
2. Either a list of per chromosome genotype files, or one file for genotype data of the entire genome. Genotype data has to be in PLINK `bed` format. 
3. Vector of lists of phenotype files per region to be analyzed, in UCSC `bed.gz` with index in `bed.gz.tbi` formats.
4. Vector of covariate files corresponding to the lists above.
5. Customized association windows file for variants (cis or trans). If it is not provided, a fixed sized window will be used around the region (a cis-window)
6. Optionally a vector of names of the phenotypic conditions in the form of `cond1 cond2 cond3` separated with whitespace.

Input 2 and 3 should be outputs from `genotype_per_region` and `annotate_coord` modules in previous preprocessing steps. 4 should be output of `covariate_preprocessing` pipeline that contains genotype PC, phenotypic hidden confounders and fixed covariates.

### Example genotype data

```
#chr        path
chr21 /mnt/mfs/statgen/xqtl_workflow_testing/protocol_example.genotype.chr21.bed
chr22 /mnt/mfs/statgen/xqtl_workflow_testing/protocol_example.genotype.chr22.bed
```

Alternatively, simply use `protocol_example.genotype.chr21_22.bed` if all chromosomes are in the same file.

### Example phenotype list

```
#chr    start   end ID  path
chr12   752578  752579  ENSG00000060237  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
chr12   990508  990509  ENSG00000082805  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
chr12   2794969 2794970 ENSG00000004478  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
chr12   4649113 4649114 ENSG00000139180  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
chr12   6124769 6124770 ENSG00000110799  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
chr12   6534516 6534517 ENSG00000111640  /home/gw/GIT/github/fungen-xqtl-analysis/analysis/Wang_Columbia/ROSMAP/MWE/output/phenotype/protocol_example.protein.bed.gz
```

### Example association-window file

It should have strictly 4 columns, with the header a commented out line:

```
#chr    start    end    gene_id
chr10   0    6480000    ENSG00000008128
chr1    0    6480000    ENSG00000008130
chr1    0    6480000    ENSG00000067606
chr1    0    7101193    ENSG00000069424
chr1    0    7960000    ENSG00000069812
chr1    0    6480000    ENSG00000078369
chr1    0    6480000    ENSG00000078808
```

The key is that the 4th column ID should match with the 4th column ID in the phenotype list. Otherwise the association-window to analyze will not be found.

### About indels

Option `--no-indel` will remove indel from analysis. FIXME: Gao need to provide more guidelines how to deal with indels in practice.

## Output

For each analysis region, the output is SuSiE model fitted and saved in RDS format.

## Minimal Working Example Steps

### i. SuSiE with TWAS weights from multiple methods

Timing [FIXME]

Below we duplicate the examples for phenotype and covariates to demonstrate that when there are multiple phenotypes for the same genotype it is possible to use this pipeline to analyze all of them (more than two is accepted as well).

Here using `--region-name` we focus the analysis on 3 genes. In practice if this parameter is dropped, the union of all regions in all phenotype region lists will be analyzed. It is possible for some of the regions there are no genotype data, in which case the pipeline will output RDS files with a warning message to indicate the lack of genotype data to analyze.

**Note:** Suggested output naming convention is cohort_modality, eg ROSMAP_snRNA_pseudobulk.

In [ ]:
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb susie_twas \
    --name test_susie_twas \
    --genoFile output/genotype_by_chrom/wgs.merged.plink_qc.1.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --phenotype-names test_pheno \
    --max-cv-variants 5000 --ld_reference_meta_file reference_data/ADSP_R4_EUR/ld_meta_file.tsv \
    --region-name ENSG00000049246 ENSG00000054116 ENSG00000116678 \
    --save-data \
    --cwd output/mnm_regression/susie_twas

It is also possible to analyze a selected list of regions using option `--region-list`. The last column of this file will be used for the list to analyze. Here for example use the same list of regions as we used for customized association-window:

In [ ]:
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb susie_twas \
    --name test_susie_twas \
    --genoFile output/genotype_by_chrom/wgs.merged.plink_qc.1.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --phenotype-names test_pheno \
    --max-cv-variants 5000 --ld_reference_meta_file reference_data/ADSP_R4_EUR/ld_meta_file.tsv \
    --region-list data/region_list.bed \
    --save-data \
    --cwd output/mnm_regression/susie_twas

**Note:** When both `--region-name` and `--region-list` are used, the union of regions from these parameters will be analyzed. 

FIXME: We should probably just explain these parameters, will work better for conversion script


To perform fine-mapping only without TWAS weights,

```
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb susie_twas --no-twas-weights ... # rest of parameters the same. 
```

To perform fine-mapping and TWAS weights without cross validation,

```
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb susie_twas --twas-cv-folds 0 ... # rest of parameters the same. 
```

It is also possible to specify a subset of samples to analyze, using `--keep-samples` parameter. For example we create a file to keep the ID of 50 samples,

In [ ]:
zcat output/covariate/protocol_example.protein.protocol_example.samples.protocol_example.genotype.chr21_22.pQTL.plink_qc.prune.pca.Marchenko_PC.gz | head -1 | awk '{for (i=2; i<=51; i++) printf $i " "; print ""}'> output/keep_samples.txt

then use them in our analysis,

```
sos run xqtl-protocol/pipeline/mnm_regression.ipynb susie_twas --keep-samples output/keep_samples.txt ... # rest of parameters the same
```

### ii. fSuSiE

Timing [FIXME]

**Note:** Suggested output naming convention is cohort_modality, eg ROSMAP_snRNA_pseudobulk.

In [ ]:
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb fsusie \
    --cwd output/fsusie/ \
    --name   Mic  \
    --genoFile data/fsusie/mwe.genotype_by_chrom_files.txt \
    --phenoFile data/fsusie/mwe.pheno.region_list \
    --covFile   data/fsusie/mwe.chr7_139293693_145380632.Marchenko_PC.anon.gz \
    --cis-window 0 --max-cv-variants 5000 \
    --susie_top_pc 0 --phenotype-names ROSMAP_Mic_snATACQTL --maf 0.01 \
    --save-data \
    --numThreads 8 \
    --post_processing "none" --small-sample-correction 

```
INFO: Running get_analysis_regions: 
Loading customized association analysis window from reference_data/TAD/TADB_enhanced_cis.bed
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: Running fsusie: 
INFO: fsusie is completed.
INFO: fsusie output:   /restricted/projectnb/xqtl/xqtl_protocol/toy_xqtl_protocol/output/fsusie/fsus/Mic.chr7_139293693_145380632.fsusie_mixture_normal_none__top_pc_weights.rds
INFO: Workflow fsusie (ID=wcb4ce30c08d43c15) is executed successfully with 2 completed steps.
```

### iii. mvSuSiE with TWAS weights from mr.mash

In [ ]:
sos run renovated_code/snakemake/modular_sos/notebooks/mnm_regression.ipynb mnm \
    --name test_mnm --cwd output/mnm \
    --genoFile output/genotype_by_chrom/wgs.merged.plink_qc.genotype_by_chrom_files.txt \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    output/covariate/bulk_rnaseq_tpm_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --region-name ENSG00000073921 --save-data --no-skip-twas-weights \
    --phenotype-names Ast Exc Inh Mic OPC Oli \
    --mixture_prior output/multivariate_mixture/MWE_ed_bovy.EE.prior.rds \
    --max_cv_variants 5000 \
	--ld_reference_meta_file data/ld_meta_file_with_bim.tsv 

```
INFO: Running [32mget_analysis_regions[0m: 
INFO: [32mget_analysis_regions[0m is [32mcompleted[0m.
INFO: [32mget_analysis_regions[0m output:   [32mregional_data[0m
INFO: Running [32mmnm[0m: 
INFO: [32mmnm[0m is [32mcompleted[0m.
INFO: [32mmnm[0m output:   [32m/restricted/projectnb/xqtl/xqtl_protocol/toy_xqtl_protocol/output/mnm/multivariate_fine_mapping/test_mnm.chr11_ENSG00000073921.multicontext_bvsr.rds /restricted/projectnb/xqtl/xqtl_protocol/toy_xqtl_protocol/output/mnm/multivariate_twas_weights/test_mnm.chr11_ENSG00000073921.multicontext_twas_weights.rds[0m
INFO: Workflow mnm (ID=w080346b338ea4ab5) is executed successfully with 2 completed steps.
```

## Troubleshooting

| Step | Substep | Problem | Possible Reason | Solution |
|------|---------|---------|------------------|---------|
|  |  |  |  |  |




## Command interface

In [ ]:
sos run mnm_regression.ipynb -h

## Workflow implementation

In [1]:
[global]
parameter: renovated_code_dir = path('renovated_code/script')  # override with --renovated-code-dir
# It is required to input the name of the analysis
parameter: name = str
parameter: cwd = path("output")
# A list of file paths for genotype data, or the genotype data itself. 
parameter: genoFile = path
# One or multiple lists of file paths for phenotype data.
parameter: phenoFile = paths
# One or multiple lists of file paths for phenotype ID mapping file. The first column should be the original ID, the 2nd column should be the ID to be mapped to.
parameter: phenoIDFile = paths()
# Covariate file path
parameter: covFile = paths
# Optional: if a region list is provide the analysis will be focused on provided region. 
# The LAST column of this list will contain the ID of regions to focus on
# Otherwise, all regions with both genotype and phenotype files will be analyzed
parameter: region_list = path()
# Optional: if a region name is provided 
# the analysis would be focused on the union of provides region list and region names
parameter: region_name = []
# Only focus on a subset of samples
parameter: keep_samples = path()
# Only focus on a subset of variants
parameter: keep_variants = path()
# An optional list documenting the custom association window for each region to analyze, with four column, chr, start, end, region ID (eg gene ID).
# If this list is not provided, the default `window` parameter (see below) will be used.
parameter: customized_association_windows = path()
# Specify the cis window for the up and downstream radius to analyze around the region of interest in units of bp
# When this is set to negative, we will rely on using customized_association_windows
parameter: cis_window = -1
# save data object or not
parameter: save_data = False
# Name of phenotypes
parameter: phenotype_names = [f'{x:bn}' for x in phenoFile]
# And indicator whether it is trans-analysis, ie, not using phenotypic coordinate information
parameter: trans_analysis = False
parameter: seed = 999

# association analysis paramters
# initial number of single effects for SuSiE
parameter: init_L = 8
# maximum number of single effects to use for SuSiE
parameter: max_L = 30
parameter: estimate_residual_method = ""
parameter: small_sample_correction = False
# remove a variant if it has more than imiss missing individual level data
parameter: imiss = 1.0
# MAF and variance of X cutoff
parameter: maf = 0.0025
parameter: xvar_cutoff = 0.0
# MAC cutoff, on top of MAF cutoff
parameter: mac = 5
# Remove indels if indel = False
parameter: indel = True
parameter: pip_cutoff = 0.025
parameter: coverage = [0.95, 0.7, 0.5]
# If this value is not 0, then an initial single effect analysis will be performed 
# to determine if follow up analysis will be continued or to simply return NULL
# If this is negative we use a default way to determine this cutoff which is conservative but still useful
parameter: skip_analysis_pip_cutoff = []
# Skip fine-mapping
parameter: skip_fine_mapping = False
# Skip TWAS weights computation
parameter: skip_twas_weights = False
# Perform K folds valiation CV for TWAS
# Set it to zero if this is to be skipped
parameter: twas_cv_folds = 5
parameter: twas_cv_threads = twas_cv_folds
# maximum number of variants to consider for CV
# We will randomly pick a subset of it for CV purpose
# We can set it to eg 8000 to save computational burden althought may risk overfitting for methods comparison purpose
# When set to -1 we don't use this feature
parameter: max_cv_variants = -1
parameter: ld_reference_meta_file = path()
# Analysis environment settings
parameter: container = ""
import re
parameter: entrypoint= ('micromamba run -a "" -n' + ' ' + re.sub(r'(_apptainer:latest|_docker:latest|\.sif)$', '', container.split('/')[-1])) if container else ""
# For cluster jobs, number commands to run per job
parameter: job_size = 200
# Wall clock time expected
parameter: walltime = "1h"
# Memory expected
parameter: mem = "20G"
# Number of threads
parameter: numThreads = 1

if len(phenoFile) != len(covFile):
    raise ValueError("Number of input phenotypes files must match that of covariates files")
if len(phenoFile) != len(phenotype_names):
    raise ValueError("Number of input phenotypes files must match the number of phenotype names")
if len(phenoIDFile) > 0 and len(phenoFile) != len(phenoIDFile):
    raise ValueError("Number of input phenotypes files must match the number of phenotype ID mapping files")

if len(skip_analysis_pip_cutoff) == 0:
    skip_analysis_pip_cutoff = [0.0] * len(phenoFile)
if len(skip_analysis_pip_cutoff) == 1:
    skip_analysis_pip_cutoff = skip_analysis_pip_cutoff * len(phenoFile)
if len(skip_analysis_pip_cutoff) != len(phenoFile):
    raise ValueError(f"``skip_analysis_pip_cutoff`` should have either length 1 or length the same as phenotype files ({len(phenoFile)} in this case)")

# make it into an R List string
skip_analysis_pip_cutoff = [f"'{y}'={x}" for x,y in zip(skip_analysis_pip_cutoff, phenotype_names)]
    
def group_by_region(lst, partition):
    # from itertools import accumulate
    # partition = [len(x) for x in partition]
    # Compute the cumulative sums once
    # cumsum_vector = list(accumulate(partition))
    # Use slicing based on the cumulative sums
    # return [lst[(cumsum_vector[i-1] if i > 0 else 0):cumsum_vector[i]] for i in range(len(partition))]
    return partition

import os
import pandas as pd

def adapt_file_path(file_path, reference_file):
    """
    Adapt a single file path based on its existence and a reference file's path.

    Args:
    - file_path (str): The file path to adapt.
    - reference_file (str): File path to use as a reference for adaptation.

    Returns:
    - str: Adapted file path.

    Raises:
    - FileNotFoundError: If no valid file path is found.
    """
    reference_path = os.path.dirname(reference_file)

    # Check if the file exists
    if os.path.isfile(file_path):
        return file_path

    # Check file name without path
    file_name = os.path.basename(file_path)
    if os.path.isfile(file_name):
        return file_name

    # Check file name in reference file's directory
    file_in_ref_dir = os.path.join(reference_path, file_name)
    if os.path.isfile(file_in_ref_dir):
        return file_in_ref_dir

    # Check original file path prefixed with reference file's directory
    file_prefixed = os.path.join(reference_path, file_path)
    if os.path.isfile(file_prefixed):
        return file_prefixed

    # If all checks fail, raise an error
    raise FileNotFoundError(f"No valid path found for file: {file_path}")

def adapt_file_path_all(df, column_name, reference_file):
    return df[column_name].apply(lambda x: adapt_file_path(x, reference_file))

In [1]:
[get_analysis_regions: shared = "regional_data"]
# input is genoFile, phenoFile, covFile and optionally region_list. If region_list presents then we only analyze what's contained in the list.
# regional_data should be a dictionary like:
#{'data': [("genotype_1.bed", "phenotype_1.bed.gz", "covariate_1.gz"), ("genotype_2.bed", "phenotype_1.bed.gz", "phenotype_2.bed.gz", "covariate_1.gz", "covariate_2.gz") ... ],
# 'meta_info': [("chr12:752578-752579","chr12:752577-752580", "gene_1", "trait_1"), ("chr13:852580-852581","chr13:852579-852580", "gene_2", "trait_1", "trait_2") ... ]}
import numpy as np

def preload_id_map(id_map_files):
    id_maps = {}
    for id_map_file in id_map_files:
        if id_map_file is not None and os.path.isfile(id_map_file):
            df = pd.read_csv(id_map_file, sep=r"\s+", header=None, comment='#', names=['old_ID', 'new_ID'])
            id_maps[id_map_file] = df.set_index('old_ID')['new_ID'].to_dict()
    return id_maps

def load_and_apply_id_map(pheno_path, id_map_path, preloaded_id_maps):
    pheno_df = pd.read_csv(pheno_path, sep=r"\s+", header=0)
    pheno_df['Original_ID'] = pheno_df['ID']
    if id_map_path in preloaded_id_maps:
        id_map = preloaded_id_maps[id_map_path]
        pheno_df['ID'] = pheno_df['ID'].map(id_map).fillna(pheno_df['ID'])
    return pheno_df

def filter_by_region_ids(data, region_ids):
    if region_ids is not None and len(region_ids) > 0:
        data = data[data['ID'].isin(region_ids)]
    return data

def custom_join(series):
    # Initialize an empty list to hold the processed items
    result = []
    for item in series:
        if ',' in item:
            # If the item contains commas, split by comma and convert to tuple
            result.append(tuple(item.split(',')))
        else:
            # If the item does not contain commas, add it directly
            result.append(item)
    # Convert the list of items to a tuple and return
    return tuple(result)

def aggregate_phenotype_data(accumulated_pheno_df):
    if not accumulated_pheno_df.empty:
        accumulated_pheno_df = accumulated_pheno_df.groupby(['#chr','ID','cond','path','cov_path'], as_index=False).agg({
            '#chr': lambda x: np.unique(x).astype(str)[0],
            'ID': lambda x: np.unique(x).astype(str)[0],
            'Original_ID': ','.join,
            'start': 'min',
            'end': 'max'
        }).groupby(['#chr','ID'], as_index=False).agg({
            'cond': ','.join,
            'path': ','.join,
            'Original_ID': custom_join,
            'cov_path': ','.join,
            'start': 'min',
            'end': 'max'
        })
    return accumulated_pheno_df

def process_cis_files(pheno_files, cov_files, phenotype_names, pheno_id_files, region_ids, preloaded_id_maps):
    '''
    Example output:
    #chr    start      end    ID  Original_ID   path     cov_path             cond
    chr12   752578   752579  ENSG00000060237  Q9H4A3,P62873  protocol_example.protein_1.bed.gz,protocol_example.protein_2.bed.gz  covar_1.gz,covar_2.gz  trait_A,trait_B
    '''
    accumulated_pheno_df = pd.DataFrame()
    pheno_id_files = [None] * len(pheno_files) if len(pheno_id_files) == 0 else pheno_id_files

    for pheno_path, cov_path, phenotype_name, id_map_path in zip(pheno_files, cov_files, phenotype_names, pheno_id_files):
        if not os.path.isfile(cov_path):
            raise FileNotFoundError(f"No valid path found for file: {cov_path}")
        pheno_df = load_and_apply_id_map(pheno_path, id_map_path, preloaded_id_maps)
        pheno_df = filter_by_region_ids(pheno_df, region_ids)
        
        if not pheno_df.empty:
            pheno_df.iloc[:, 4] = adapt_file_path_all(pheno_df, pheno_df.columns[4], f"{pheno_path:a}")
            pheno_df = pheno_df.assign(cov_path=str(cov_path), cond=phenotype_name)           
            accumulated_pheno_df = pd.concat([accumulated_pheno_df, pheno_df], ignore_index=True)

    accumulated_pheno_df = aggregate_phenotype_data(accumulated_pheno_df)
    return accumulated_pheno_df

def process_trans_files(pheno_files, cov_files, phenotype_names, pheno_id_files, region_ids, customized_association_windows):
    '''
    Example output:
    #chr    start      end    ID  Original_ID   path     cov_path             cond
    chr21   0   0  chr21_18133254_19330300  carnitine,benzoate,hippurate  metabolon_1.bed.gz,metabolon_2.bed.gz  covar_1.gz,covar_2.gz  trait_A,trait_B
    '''
    
    if not os.path.isfile(customized_association_windows):
        raise ValueError("Customized association analysis window must be specified for trans analysis.")
    accumulated_pheno_df = pd.DataFrame()
    pheno_id_files = [None] * len(pheno_files) if len(pheno_id_files) == 0 else pheno_id_files
    genotype_windows = pd.read_csv(customized_association_windows, comment="#", header=None, names=["#chr","start","end","ID"], sep="\t")
    genotype_windows = filter_by_region_ids(genotype_windows, region_ids)
    if genotype_windows.empty:
        return accumulated_pheno_df
    
    for pheno_path, cov_path, phenotype_name, id_map_path in zip(pheno_files, cov_files, phenotype_names, pheno_id_files):
        if not os.path.isfile(cov_path):
            raise FileNotFoundError(f"No valid path found for file: {cov_path}")
        pheno_df = pd.read_csv(pheno_path, sep=r"\s+", header=0, names=['Original_ID', 'path'])
        if not pheno_df.empty:
            pheno_df.iloc[:, -1] = adapt_file_path_all(pheno_df, pheno_df.columns[-1], f"{pheno_path:a}")
            pheno_df = pheno_df.assign(cov_path=str(cov_path), cond=phenotype_name)
            # Here we combine genotype_windows which contains "#chr" and "ID" to pheno_df by creating a cartesian product
            pheno_df = pd.merge(genotype_windows.assign(key=1), pheno_df.assign(key=1), on='key').drop('key', axis=1)
            # Only set start and end columns to zero if they don't exist
            if 'start' not in pheno_df.columns:
                pheno_df['start'] = 0
            if 'end' not in pheno_df.columns:
                pheno_df['end'] = 0
            if id_map_path is not None:
                # Filter pheno_df by specific association-window and phenotype pairs
                association_analysis_pair = pd.read_csv(id_map_path, sep=r"\s+", header=None, comment='#', names=['ID', 'Original_ID'])
                pheno_df = pd.merge(pheno_df, association_analysis_pair, on=['ID', 'Original_ID'])
            accumulated_pheno_df = pd.concat([accumulated_pheno_df, pheno_df], ignore_index=True)

    accumulated_pheno_df = aggregate_phenotype_data(accumulated_pheno_df)
    return accumulated_pheno_df

# Load genotype meta data
if f"{genoFile:x}" == ".bed":
    geno_meta_data = pd.DataFrame([("chr"+str(x), f"{genoFile:a}") for x in range(1,23)] + [("chrX", f"{genoFile:a}")], columns=['#chr', 'geno_path'])
else:
    geno_meta_data = pd.read_csv(f"{genoFile:a}", sep =r"\s+", header=0)
    geno_meta_data.iloc[:, 1] = adapt_file_path_all(geno_meta_data, geno_meta_data.columns[1], f"{genoFile:a}")
    geno_meta_data.columns = ['#chr', 'geno_path']
    geno_meta_data['#chr'] = geno_meta_data['#chr'].apply(lambda x: str(x) if str(x).startswith('chr') else f'chr{x}')

# Checking the DataFrame
valid_chr_values = [f'chr{x}' for x in range(1, 23)] + ['chrX']
if not all(value in valid_chr_values for value in geno_meta_data['#chr']):
    raise ValueError("Invalid chromosome values found. Allowed values are chr1 to chr22 and chrX.")

region_ids = []
# If region_list is provided, read the file and extract IDs
if region_list.is_file():
    region_list_df = pd.read_csv(region_list, delim_whitespace=True, header=None, comment = "#")
    region_ids = region_list_df.iloc[:, -1].unique()  # Extracting the last column for IDs

# If region_name is provided, include those IDs as well
# --region-name A B C will result in a list of ["A", "B", "C"] here
if len(region_name) > 0:
    region_ids = list(set(region_ids).union(set(region_name)))

if trans_analysis:
    meta_data = process_trans_files(phenoFile, covFile, phenotype_names, phenoIDFile, region_ids, customized_association_windows)
else:
    meta_data = process_cis_files(phenoFile, covFile, phenotype_names, phenoIDFile, region_ids, preload_id_map(phenoIDFile))
    
if not meta_data.empty:
    meta_data = meta_data.merge(geno_meta_data, on='#chr', how='inner')
    # Adjust association-window
    if os.path.isfile(customized_association_windows):
        print(f"Loading customized association analysis window from {customized_association_windows}")
        association_windows_list = pd.read_csv(customized_association_windows, comment="#", header=None, names=["#chr","start","end","ID"], sep="\t")
        meta_data = pd.merge(meta_data, association_windows_list, on=['#chr', 'ID'], how='left', suffixes=('', '_association'))
        mismatches = meta_data[meta_data['start_association'].isna()]
        if not mismatches.empty:
            print("First 5 mismatches:")
            print(mismatches[['ID']].head())
            raise ValueError(f"{len(mismatches)} regions to analyze cannot be found in ``{customized_association_windows}``. Please check your ``{customized_association_windows}`` database to make sure it contains all association-window definitions. ")
    else:
        if cis_window < 0 :
            raise ValueError("Please either input valid path to association-window file via ``--customized-association-windows``, or set ``--cis-window`` to a non-negative integer.")
        if cis_window == 0:
            print("Warning: only variants within the range of start and end of molecular phenotype will be considered since cis_window is set to zero and no customized association window file was found. Please make sure this is by design.")
        meta_data['start_association'] = meta_data['start'].apply(lambda x: max(x - cis_window, 0))
        meta_data['end_association'] = meta_data['end'] + cis_window

    # Example meta_data:
    # #chr    start      end    start_association       end_association           ID  Original_ID   path     cov_path             cond             coordinate     geno_path
    # 0  chr12   752578   752579  652578   852579  ENSG00000060237  Q9H4A3,P62873  protocol_example.protein_1.bed.gz,protocol_example.protein_2.bed.gz  covar_1.gz,covar_2.gz  trait_A,trait_B    chr12:752578-752579  protocol_example.genotype.chr21_22.bed       
    # Create the final dictionary
    regional_data = {
        'data': [(row['geno_path'], *row['path'].split(','), *row['cov_path'].split(',')) for _, row in meta_data.iterrows()],
        'meta_info': [(f"{row['#chr']}:{row['start']}-{row['end']}", # this is the phenotypic region to extract data from
                       f"{row['#chr']}:{row['start_association']}-{row['end_association']}", # this is the association window region
                       row['ID'], row['Original_ID'], *row['cond'].split(',')) for _, row in meta_data.iterrows()]
    }
else:
    regional_data = {'data':[], 'meta_info':[]}

## Univariate analysis: SuSiE and TWAS

In [ ]:
[susie_twas]
# Further limit TWAS to only using selected variants
parameter: min_twas_maf = 0.01
parameter: min_twas_xvar = 0.01
depends: sos_variable("regional_data")
# Check if both 'data' and 'meta_info' are empty lists
stop_if(len(regional_data['data']) == 0, f'Either genotype or phenotype data are not available for region {", ".join(region_name)}.')
meta_info = regional_data['meta_info']
input: regional_data["data"], group_by = lambda x: group_by_region(x, regional_data["data"]), group_with = "meta_info"

if skip_fine_mapping and skip_twas_weights:
    save_data = True
    output_files = [f'{cwd:a}/data/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.univariate_data.rds']
elif not skip_fine_mapping and skip_twas_weights:
    output_files = [f'{cwd:a}/fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.univariate_bvsr.rds']
elif skip_fine_mapping and not skip_twas_weights:
    output_files = [f'{cwd:a}/twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.univariate_twas_weights.rds']
else:
    output_files = [f'{cwd:a}/fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.univariate_bvsr.rds',
    f'{cwd:a}/twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.univariate_twas_weights.rds']
output: output_files
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bnn}'
bash: expand= "${ }", stderr = f"{_output[0]:nn}.susie_twas.stderr", stdout = f"{_output[0]:nn}.susie_twas.stdout", container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/mnm_analysis/mnm_methods/susie_twas.R \
        --genotype ${_input[0]:a} \
        --phenotype "${",".join([str(x.absolute()) for x in _input[1:len(_input)//2+1]])}" \
        --covariate "${",".join([str(x.absolute()) for x in _input[len(_input)//2+1:]])}" \
        --region "${_meta_info[0]}" \
        --window "${_meta_info[1]}" \
        --region-name "${_meta_info[2]}" \
        --extract-region-names "${"|".join([x if isinstance(x,str) else ",".join(x) for x in _meta_info[3]])}" \
        --conditions "${",".join(_meta_info[4:])}" \
        --skip-analysis-pip-cutoff "${",".join(skip_analysis_pip_cutoff)}" \
        --maf ${maf} \
        --mac ${mac} \
        --imiss ${imiss} \
        ${"--indel" if indel else ""} \
        ${"--keep-samples " + str(keep_samples) if keep_samples.is_file() else ""} \
        ${"--keep-variants " + str(keep_variants) if not keep_variants.is_dir() else ""} \
        ${"--save-data" if save_data else ""} \
        --pip-cutoff ${pip_cutoff} \
        --coverage "${",".join([str(x) for x in coverage])}" \
        --seed ${seed} \
        --cwd ${cwd:a} \
        ${"--skip-fine-mapping" if skip_fine_mapping else ""} \
        ${"--skip-twas-weights" if skip_twas_weights else ""} \
        ${"--trans-analysis" if trans_analysis else ""} \
        --init-l ${init_L} \
        --max-l ${max_L} \
        ${"--estimate-residual-method " + str(estimate_residual_method) if estimate_residual_method else ""} \
        ${"--small-sample-correction" if small_sample_correction else ""} \
        --max-cv-variants ${max_cv_variants} \
        --twas-cv-folds ${twas_cv_folds} \
        --twas-cv-threads ${twas_cv_threads} \
        --min-twas-maf ${min_twas_maf} \
        --min-twas-xvar ${min_twas_xvar} \
        ${"--ld-reference-meta-file " + str(ld_reference_meta_file) if not ld_reference_meta_file.is_dir() else ""} \
        --output-prefix ${_output[0]:nn}


## Multivariate analysis: mvSuSiE and mr.mash

In [ ]:
[mnm]
# Prior model file generated from mashr. 
# Default will be used if it does not exist.
parameter: mixture_prior = path()
parameter: mixture_prior_cv = path()
parameter: prior_weights_min = 5e-4
parameter: prior_canonical_matrices = False
parameter: sample_partition = path() 
parameter: mvsusie_max_iter = 200
parameter: mrmash_max_iter = 5000
depends: sos_variable("regional_data")
# Check if both 'data' and 'meta_info' are empty lists
stop_if(len(regional_data['data']) == 0, f'Either genotype or phenotype data are not available for region {", ".join(region_name)}.')

meta_info = regional_data['meta_info']

input: regional_data["data"], group_by = lambda x: group_by_region(x, regional_data["data"]), group_with = "meta_info"
if skip_fine_mapping and skip_twas_weights:
    save_data = True
    output_files = [f'{cwd:a}/data/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_data.rds']
elif not skip_fine_mapping and skip_twas_weights:
    output_files = [f'{cwd:a}/multivariate_fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_bvsr.rds']
elif skip_fine_mapping and not skip_twas_weights:
    # Warning that fine-mapping is required for TWAS weights
    if not ld_reference_meta_file.is_dir():
        print("WARNING: Fine-mapping is required to generate TWAS weights. Setting skip_fine_mapping to False. "
              "Fine-mapping will be performed and saved on variants listed in %s." % ld_reference_meta_file)
    else:
        print("WARNING: Fine-mapping is required to generate TWAS weights. Setting skip_fine_mapping to False.")
    skip_fine_mapping = False
    output_files = [f'{cwd:a}/multivariate_fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_bvsr.rds',
                    f'{cwd:a}/multivariate_twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_twas_weights.rds']
else:    
    output_files = [f'{cwd:a}/multivariate_fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_bvsr.rds',
                    f'{cwd:a}/multivariate_twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multicontext_twas_weights.rds']   
output: output_files
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
bash: expand= "${ }", stderr = f"{_output[0]:nn}.mnm.stderr", stdout = f"{_output[0]:nn}.mnm.stdout", container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/mnm_analysis/mnm_methods/mnm.R \
        --genotype ${_input[0]:a} \
        --phenotype "${",".join([str(x.absolute()) for x in _input[1:len(_input)//2+1]])}" \
        --covariate "${",".join([str(x.absolute()) for x in _input[len(_input)//2+1:]])}" \
        --region "${_meta_info[0]}" \
        --window "${_meta_info[1]}" \
        --region-name "${_meta_info[2]}" \
        --extract-region-names "${"|".join([x if isinstance(x,str) else ",".join(x) for x in _meta_info[3]])}" \
        --conditions "${",".join(_meta_info[4:])}" \
        --skip-analysis-pip-cutoff "${",".join(skip_analysis_pip_cutoff)}" \
        --maf ${maf} \
        --mac ${mac} \
        --imiss ${imiss} \
        ${"--indel" if indel else ""} \
        ${"--keep-samples " + str(keep_samples) if keep_samples.is_file() else ""} \
        ${"--keep-variants " + str(keep_variants) if not keep_variants.is_dir() else ""} \
        ${"--save-data" if save_data else ""} \
        --pip-cutoff ${pip_cutoff} \
        --coverage "${",".join([str(x) for x in coverage])}" \
        --seed ${seed} \
        --cwd ${cwd:a} \
        ${"--skip-fine-mapping" if skip_fine_mapping else ""} \
        ${"--skip-twas-weights" if skip_twas_weights else ""} \
        --xvar-cutoff ${xvar_cutoff} \
        --mvsusie-max-iter ${mvsusie_max_iter} \
        --mrmash-max-iter ${mrmash_max_iter} \
        --max-cv-variants ${max_cv_variants} \
        --twas-cv-folds ${twas_cv_folds} \
        --twas-cv-threads ${twas_cv_threads} \
        ${"--ld-reference-meta-file " + str(ld_reference_meta_file) if not ld_reference_meta_file.is_dir() else ""} \
        ${"--mixture-prior " + str(mixture_prior) if mixture_prior.is_file() else ""} \
        ${"--mixture-prior-cv " + str(mixture_prior_cv) if mixture_prior_cv.is_file() else ""} \
        --prior-weights-min ${prior_weights_min} \
        ${"--prior-canonical-matrices" if prior_canonical_matrices else ""} \
        ${"--sample-partition " + str(sample_partition) if sample_partition.is_file() else ""} \
        --output-prefix ${_output[0]:nn}


## xQTL fine-mapping across multiple genes

This pipeline jointly fine-map multiple genes in a given region. To determine if a gene is eligable for joint fine-mapping with other genes, it will first load the univeriate fine-mapping result of a gene and assess if its top_loci candidates overlap with any of the other genes. Genes don't statisfy this requirement will be dropped. It is therefore required that the univariate fine-mapping results and its meta data are available, in the format of `susie_twas` output.

In [ ]:
[mnm_genes]
depends: sos_variable("regional_data")
# Check if both 'data' and 'meta_info' are empty lists
stop_if(len(regional_data['data']) == 0, f'Either genotype or phenotype data are not available for region {", ".join(region_name)}.')
 
meta_info = regional_data['meta_info']
genes = meta_info[0][3]
if isinstance(genes, tuple):
    genes = genes[0] if isinstance(genes[0], tuple) else list(genes)
else:
    genes = [genes]

if len(skip_analysis_pip_cutoff) == 0:
    skip_analysis_pip_cutoff = [0.0] * len(genes)

if len(skip_analysis_pip_cutoff) == 1:
    skip_analysis_pip_cutoff = skip_analysis_pip_cutoff * len(genes)

skip_analysis_pip_cutoff = ["'{}'={}".format(genes[i], skip_analysis_pip_cutoff[i].split('=')[1]) for i in range(len(genes))]

parameter: pheno_id_map_file = path
parameter: fine_mapping_meta = path
parameter: data_driven_prior = False
# Parameters below are relevant when we use data driven prior
# which is an experimental feature
parameter: n_random = 10
parameter: n_null = 10
parameter: independent_variant_list = path()
parameter: prior_weights_min = 5e-4
parameter: prior_canonical_matrices = False
# This is relevant to cross-validation
parameter: sample_partition = path() 
parameter: mvsusie_max_iter = 200
parameter: mrmash_max_iter = 5000

if not data_driven_prior:
     prior_canonical_matrices = True
input: regional_data["data"],group_by = lambda x: group_by_region(x, regional_data["data"]), group_with = "meta_info"

if skip_fine_mapping and skip_twas_weights:
    save_data = True
    output_files = [f'{cwd:a}/data/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multigene_data.rds']
elif not skip_fine_mapping and skip_twas_weights:
    output_files = [f'{cwd:a}/multivariate_fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multigene_bvsr.rds']
elif skip_fine_mapping and not skip_twas_weights:
    output_files = [f'{cwd:a}/multivariate_twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multigene_twas_weights.rds']
else:    
    output_files = [f'{cwd:a}/multivariate_fine_mapping/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multigene_bvsr.rds',
    f'{cwd:a}/multivariate_twas_weights/{name}.{_meta_info[0].split(":")[0]}_{_meta_info[2]}.multigene_twas_weights.rds']    
output: output_files
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand= "${ }", stderr = f"{_output[0]:nn}.mnm_genes.stderr", stdout = f"{_output[0]:nn}.mnm_genes.stdout", container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/mnm_analysis/mnm_methods/mnm_genes.R \
        --genotype ${_input[0]:a} \
        --phenotype "${",".join([str(x.absolute()) for x in _input[1:len(_input)//2+1]])}" \
        --covariate "${",".join([str(x.absolute()) for x in _input[len(_input)//2+1:]])}" \
        --region-coord "${_meta_info[0]}" \
        --grange "${_meta_info[1]}" \
        --region-name "${_meta_info[2]}" \
        --raw-region-names "${"|".join([x if isinstance(x,str) else ",".join(x) for x in _meta_info[3]])}" \
        --condition-name "${_meta_info[-1]}" \
        --skip-analysis-pip-cutoff "${",".join(skip_analysis_pip_cutoff)}" \
        --pheno-id-map-file ${pheno_id_map_file:r} \
        --fine-mapping-meta ${fine_mapping_meta:r} \
        --maf ${maf} \
        --mac ${mac} \
        --imiss ${imiss} \
        ${"--indel" if indel else ""} \
        ${"--keep-samples " + str(keep_samples) if keep_samples.is_file() else ""} \
        ${"--keep-variants " + str(keep_variants) if not keep_variants.is_dir() else ""} \
        ${"--data-driven-prior" if data_driven_prior else ""} \
        --n-random ${n_random} \
        --n-null ${n_null} \
        ${"--independent-variant-list " + str(independent_variant_list) if independent_variant_list.is_file() else ""} \
        --prior-weights-min ${prior_weights_min} \
        ${"--prior-canonical-matrices" if prior_canonical_matrices else ""} \
        ${"--sample-partition " + str(sample_partition) if sample_partition.is_file() else ""} \
        --mvsusie-max-iter ${mvsusie_max_iter} \
        --mrmash-max-iter ${mrmash_max_iter} \
        ${"--save-data" if save_data else ""} \
        ${"--skip-fine-mapping" if skip_fine_mapping else ""} \
        ${"--skip-twas-weights" if skip_twas_weights else ""} \
        --pip-cutoff ${pip_cutoff} \
        --coverage "${",".join([str(x) for x in coverage])}" \
        --max-cv-variants ${max_cv_variants} \
        --twas-cv-folds ${twas_cv_folds} \
        --twas-cv-threads ${twas_cv_threads} \
        --seed ${seed} \
        --output-files "${",".join([str(x) for x in _output])}" \
        --cwd ${cwd:a}


## Functional regression fSuSiE for epigenomic QTL fine-mapping

In [ ]:
[fsusie]
# prior can be either of ["mixture_normal", "mixture_normal_per_scale"]
parameter: prior = "mixture_normal"
parameter: max_SNP_EM = 100
# Max scale is such that 2^max_scale being the number of phenotypes in the transformed space. Default to 2^10  = 1024. Don't change it unless you know what you are doing. Max_scale should be at least larger than 5.
parameter:  max_scale = 10
# Purity and coverage used to call cs
parameter:  min_purity = 0.5
# Epigenetics mark filter
parameter: epigenetics_mark_treshold = 16
# Run susie for top pc of the fsusie input
parameter: susie_top_pc = 10
# Run fsusie with this post-processing option. Available options are TI, HMM, and none. TI is recommended as default. HMM performs particularly well when analyzing data with, say, few sampling points (i.e., ncol(Y)<30) or when the data are particularly noisy (low signal-to-noise ratio).
parameter: post_processing =  "TI"
# Run fsusie with small sample correction 
parameter: small_sample_correction = False

depends: sos_variable("regional_data")
# Check if both 'data' and 'meta_info' are empty lists
stop_if(len(regional_data['data']) == 0, f'Either genotype or phenotype data are not available for region {", ".join(region_name)}.')
meta_info = regional_data['meta_info']
input: regional_data["data"], group_by = lambda x: group_by_region(x, regional_data["data"]), group_with = "meta_info"
output: f'{cwd:a}/{step_name[:-2]}/{name}.{_meta_info[0].replace(":", "_").replace("-", "_")}.fsusie_{prior}_{post_processing}_{"_top_pc_weights" if not skip_twas_weights else ""}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand= "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/mnm_analysis/mnm_methods/fsusie.R \
        --genotype ${_input[0]:a} \
        --phenotype "${",".join([str(x.absolute()) for x in _input[1:len(_input)//2+1]])}" \
        --covariate "${",".join([str(x.absolute()) for x in _input[len(_input)//2+1:]])}" \
        --region "${_meta_info[0]}" \
        --window "${_meta_info[1]}" \
        --region-name "${_meta_info[2]}" \
        --conditions "${",".join(_meta_info[4:])}" \
        --maf ${maf} \
        --mac ${mac} \
        --imiss ${imiss} \
        ${"--indel" if indel else ""} \
        ${"--keep-samples " + str(keep_samples) if keep_samples.is_file() else ""} \
        ${"--keep-variants " + str(keep_variants) if not keep_variants.is_dir() else ""} \
        --prior "${prior}" \
        --max-snp-em ${max_SNP_EM} \
        --max-scale ${max_scale} \
        --min-purity ${min_purity} \
        --epigenetics-mark-threshold ${epigenetics_mark_treshold} \
        --susie-top-pc ${susie_top_pc} \
        --post-processing "${post_processing}" \
        ${"--small-sample-correction" if small_sample_correction else ""} \
        --pip-cutoff ${pip_cutoff} \
        --coverage "${",".join([str(x) for x in coverage])}" \
        --init-l ${init_L} \
        --max-l ${max_L} \
        ${"--skip-twas-weights" if skip_twas_weights else ""} \
        --max-cv-variants ${max_cv_variants} \
        --twas-cv-folds ${twas_cv_folds} \
        --twas-cv-threads ${twas_cv_threads} \
        ${"--save-data" if save_data else ""} \
        --output-prefix ${_output:n} \
        --cwd ${cwd:a}


## Functional regression fSuSiE with other modality

**This is still WIP --- mvfSuSiE package is still being developed**

In [ ]:
[mvfsusie]
# prior can be either of ["mixture_normal", "mixture_normal_per_scale"]
parameter: prior  = "mixture_normal_per_scale"
parameter: max_SNP_EM = 1000
depends: sos_variable("regional_data")
# Check if both 'data' and 'meta_info' are empty lists
stop_if(len(regional_data['data']) == 0, f'Either genotype or phenotype data are not available for region {", ".join(region_name)}.')

meta_info = regional_data['meta_info']
input: regional_data["data"], group_by = lambda x: group_by_region(x, regional_data["data"]), group_with = "meta_info"
output: f'{cwd:a}/{step_name[:-2]}/{name}.{_meta_info[0]}.mvfsusie_{prior}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
bash: expand= "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container, entrypoint = entrypoint
    Rscript ${renovated_code_dir}/mnm_analysis/mnm_methods/mvfsusie.R \
        --genotype ${_input[0]:a} \
        --phenotype "${",".join([str(x.absolute()) for x in _input[1::2]])}" \
        --covariate "${",".join([str(x.absolute()) for x in _input[2::2]])}" \
        --region "${_meta_info[1]}:${_meta_info[2]}-${_meta_info[3]}" \
        --maf ${maf} \
        --mac ${mac} \
        --imiss ${imiss} \
        --max-l ${max_L} \
        --output ${_output:a}
